# TRACK B2 — E4 on FLUX.1-dev @ 1024

B1 selected **FLUX.1-dev** on the pre-registered rule: highest device-specific VAE retention
(η = 0.554) among models clearing the 0.75 post-reconstruction AUC gate. Note for the paper that
**the gate excluded nobody** — even SD-1.5 cleared it at 0.813 — so the rule was not
reverse-engineered from the outcome.

**96 GB VRAM removes the memory constraint** that would otherwise have forced 512px or the
distilled schnell variant. This runs at **1024, with the base study's fingerprints and measurement
unchanged.**

### What differs from SD-3.5, and where a hand-written loop goes wrong
1. **Packed latents.** FLUX patchifies 2×2, so (B,16,128,128) → (B,4096,64). Predictions must be
   unpacked before the loss.
2. **Dynamic shift.** The sigma schedule depends on sequence length via `calculate_shift`; a fixed
   schedule trains at the wrong noise levels.
3. **Guidance embedding.** FLUX-dev is guidance-distilled and the transformer *requires* a
   `guidance` input.
4. **Three text inputs** — CLIP pooled, T5 sequence, and `text_ids`.

**F0 is a 5-step smoke test.** It surfaces any shape or dtype error in under a minute instead of
thirty. Run it before F1 and do not skip it.

### Scope
E4 asks **does the null replicate**, not what the bound is on system two. 3 seeds × 2 arms = 6
adapters; the clustered bound is reported and explicitly flagged as n = 3. This is a
**cross-system robustness test, not an architecture ablation** — VAE, backbone, latent
dimensionality, text encoder and scale all co-vary.

In [ ]:
# %% F-CFG
import os, sys, json, glob, math, time, gc, shutil, hashlib, subprocess, random, copy
from dataclasses import dataclass, asdict
from concurrent.futures import ThreadPoolExecutor
import csv as _csv
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"]=(userdata.get("HF_TOKEN") or "").strip()
except Exception: pass
from google.colab import drive; drive.mount("/content/drive", force_remount=False)
DRIVE="/content/drive/MyDrive"
if not os.path.isdir(DRIVE): raise SystemExit("Drive not mounted")

@dataclass
class CFG:
    STAGES: tuple = ("F0",)                       # F0 smoke -> F1 train -> F2 gen -> F3 meas -> F4
    BASE: str = "inv_channel/E_INV_P0_v3"
    OUT:  str = "inv_channel/E_TRACKB2_FLUX"
    MODEL: str = "black-forest-labs/FLUX.1-dev"
    CROP: int = 1024
    SEEDS: tuple = (0,1,2)
    ARMS: tuple = ("A","B")
    RANK: int = 16
    TARGETS: tuple = ("to_q","to_k","to_v","to_out.0")
    LR: float = 1e-4; STEPS: int = 2000; BATCH: int = 1; GRAD_ACC: int = 4
    CAPTION: str = "a photograph, sks style"
    GUIDANCE_TRAIN: float = 1.0                   # FLUX-dev is guidance-distilled
    G_PER: int = 500; GEN_SEED_BASE: int = 770000
    GEN_STEPS: int = 28; GEN_GUIDANCE: float = 3.5; GEN_BATCH: int = 2
    N_PERM: int = 10000; ALPHA: float = 0.01
    GRAD_CKPT: bool = False                       # 96 GB — off unless OOM
C=CFG(); MEAS=C.CROP
CFG_SHA=hashlib.sha256(json.dumps({k:v for k,v in asdict(C).items() if k!="STAGES"},
                                  sort_keys=True,default=str).encode()).hexdigest()
BASE=os.path.join(DRIVE,C.BASE); ROOT=os.path.join(DRIVE,C.OUT)
for d in ["","adapters","gens","csv","meta","train_png","figs"]: os.makedirs(os.path.join(ROOT,d),exist_ok=True)
CACHE="/content/f_cache"; os.makedirs(CACHE,exist_ok=True)
def stage(n):
    if n in C.STAGES: return True
    print(f"[{n}] SKIPPED"); return False

subprocess.run([sys.executable,"-m","pip","install","-q","PyWavelets","diffusers>=0.31",
                "transformers","peft","accelerate","safetensors","sentencepiece","protobuf"],check=False)
import numpy as np, pandas as pd, pywt, torch
import torch.nn.functional as tF
from PIL import Image
from scipy import stats as sps
try:
    import peft.import_utils as _piu; _piu.is_torchao_available=lambda: False
    import peft.tuners.lora.torchao as _plt; _plt.is_torchao_available=lambda: False
    print("[ENV] peft torchao dispatcher disabled")
except Exception: pass
Image.MAX_IMAGE_PIXELS=None
DEV="cuda" if torch.cuda.is_available() else "cpu"
DT=torch.bfloat16 if DEV=="cuda" and torch.cuda.is_bf16_supported() else torch.float32
torch.backends.cudnn.allow_tf32=False; torch.backends.cuda.matmul.allow_tf32=False
np.seterr(divide="raise",invalid="raise"); _MD=DEV
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login; login(token=os.environ["HF_TOKEN"])
else: print("[ENV] HF_TOKEN missing — FLUX.1-dev is GATED and will fail to download")
def HALT(m): raise SystemExit(f"\n{'='*70}\n[HALT] {m}\n{'='*70}")
if DT==torch.float16: HALT("fp16 fallback — bf16 GPU required")
if DEV=="cuda":
    vram=torch.cuda.get_device_properties(0).total_memory/2**30
    print(f"[ENV] {torch.cuda.get_device_name(0)}  {vram:.0f} GB  dtype {DT}")
    if vram<45: print("[ENV] WARNING: <45 GB. FLUX rank-16 LoRA at 1024 may OOM. "
                      "Set C.GRAD_CKPT=True, or drop to 512 and re-estimate fingerprints.")
def append_row(p,h,r):
    new=not os.path.exists(p)
    with open(p,"a") as f:
        if new: f.write(",".join(h)+"\n")
        f.write(",".join(str(x) for x in r)+"\n")
def done_keys(p,cols):
    if not os.path.exists(p): return set()
    with open(p) as f: return {tuple(r[k] for k in cols) for r in _csv.DictReader(f)}
def ajson(o,p,**kw):
    t=p+".tmp"; json.dump(o,open(t,"w"),**kw); os.replace(t,p)
def all_arms(): return [(f"{a}_raw_s{s}_flux",a,s) for s in C.SEEDS for a in C.ARMS]
def all_tags(): return [t for t,_,_ in all_arms()]
print(f"protocol SHA {CFG_SHA[:16]} | {len(all_arms())} adapters | MEAS {MEAS}")

In [ ]:
# %% forensic core — byte-identical to the base study, and the base fingerprints reused
def _cp(fp): return os.path.join(CACHE,hashlib.sha1(f"{fp}|{MEAS}".encode()).hexdigest()[:16]+".npy")
def load_lum(fp,cache=True):
    if cache and os.path.exists(_cp(fp)): return np.load(_cp(fp))
    with Image.open(fp) as im:
        im=im.convert("RGB"); W,H=im.size
        a=np.asarray(im.crop(((W-MEAS)//2,(H-MEAS)//2,(W-MEAS)//2+MEAS,(H-MEAS)//2+MEAS)),np.float32)
    l=(0.299*a[...,0]+0.587*a[...,1]+0.114*a[...,2]).astype(np.float32)
    if cache: np.save(_cp(fp),l)
    return l
def load_rgb(fp):
    with Image.open(fp) as im:
        im=im.convert("RGB"); W,H=im.size
        return np.asarray(im.crop(((W-MEAS)//2,(H-MEAS)//2,(W-MEAS)//2+MEAS,(H-MEAS)//2+MEAS)),np.float32)
def _c2(x,k):
    xt=torch.from_numpy(np.ascontiguousarray(x))[None,None]; kt=torch.from_numpy(k)[None,None]
    if _MD=="cuda": return tF.conv2d(xt.cuda(),kt.cuda(),padding=k.shape[0]//2)[0,0].cpu().numpy()
    return tF.conv2d(xt,kt,padding=k.shape[0]//2)[0,0].numpy()
def wres(img):
    s0=np.float32(4.0)
    co=pywt.wavedec2(img,"db8",level=4,mode="periodization"); out=[co[0]*0.0]
    for (cH,cV,cD) in co[1:]:
        bnd=[]
        for ch in (cH,cV,cD):
            ch=ch.astype(np.float32); vm=None
            for w in (3,5,7,9):
                v=np.maximum(_c2(ch*ch,np.ones((w,w),np.float32)/(w*w))-s0,0.0)
                vm=v if vm is None else np.minimum(vm,v)
            bnd.append(ch*(s0/(vm+s0)))
        out.append(tuple(bnd))
    n=pywt.waverec2(out,"db8",mode="periodization")[:img.shape[0],:img.shape[1]]
    r=n-n.mean(); sd=float(r.std())
    if not np.isfinite(sd) or sd==0: HALT("degenerate residual")
    return (r/sd).astype(np.float32)
def nccb(a,B):
    a=a-a.mean(); na=a.norm()
    Bc=B-B.mean(dim=(1,2),keepdim=True); nb=Bc.flatten(1).norm(dim=1)
    return (Bc*a).flatten(1).sum(dim=1)/(na*nb).clamp_min(1e-30)
MAN={}
for r in _csv.DictReader(open(os.path.join(BASE,"manifest","manifest.csv"))):
    MAN.setdefault((r["role"],r["split"]),[]).append(r["path"])
KFP={r:np.load(os.path.join(BASE,"fingerprints",f"K_{r}_E2.npy")) for r in ("A","B")}
print("base manifest + fingerprints reused; T split:",len(MAN[("A","T")]),len(MAN[("B","T")]))

def tdir(role): return os.path.join(ROOT,"train_png",role)
for role in C.ARMS:
    d=tdir(role); os.makedirs(d,exist_ok=True)
    if len(glob.glob(os.path.join(d,"*.png")))<len(MAN[(role,"T")]):
        for i,fp in enumerate(MAN[(role,"T")]):
            Image.fromarray(load_rgb(fp).astype(np.uint8)).save(
                os.path.join(d,f"{i:04d}.png"),compress_level=1)
        print(f"materialised {role}: {len(MAN[(role,'T')])} PNGs @ {MEAS}")

In [ ]:
# %% FLUX helpers — the four places a hand-written loop goes wrong
from diffusers import FluxPipeline

def flux_load(with_text=True):
    kw=dict(torch_dtype=DT, token=True)
    if not with_text:
        kw.update(text_encoder=None, text_encoder_2=None, tokenizer=None, tokenizer_2=None)
    return FluxPipeline.from_pretrained(C.MODEL, **kw).to(DEV)

def cached_prompt():
    p=os.path.join(ROOT,"adapters","_prompt.pt")
    if os.path.exists(p):
        z=torch.load(p,map_location=DEV)
        return z["pe"].to(DEV), z["pp"].to(DEV), z["ti"].to(DEV)
    pipe=flux_load(True)
    with torch.no_grad():
        pe,pp,ti=pipe.encode_prompt(prompt=C.CAPTION,prompt_2=C.CAPTION,
                                    device=DEV,num_images_per_prompt=1,max_sequence_length=512)
    torch.save({"pe":pe.cpu(),"pp":pp.cpu(),"ti":ti.cpu()},p)
    del pipe; gc.collect(); torch.cuda.empty_cache()
    return pe,pp,ti

def calc_shift(seq_len, base_seq=256, max_seq=4096, base_shift=0.5, max_shift=1.15):
    m=(max_shift-base_shift)/(max_seq-base_seq); b=base_shift-m*base_seq
    return seq_len*m+b

def build_sigmas(sched, seq_len):
    """FLUX uses DYNAMIC shifting: the sigma schedule depends on sequence length."""
    s=copy.deepcopy(sched)
    n=int(s.config.num_train_timesteps)
    sig=np.linspace(1.0,1/n,n)
    if getattr(s.config,"use_dynamic_shifting",False):
        mu=calc_shift(seq_len)
        sig=math.exp(mu)/(math.exp(mu)+(1/sig-1)**1.0)
    else:
        sh=float(getattr(s.config,"shift",3.0)); sig=sh*sig/(1+(sh-1)*sig)
    return torch.from_numpy(np.asarray(sig,dtype=np.float32)).to(DEV)

def pack(x):
    """(B,C,H,W) -> (B, H/2*W/2, C*4)"""
    B,Cc,H,W=x.shape
    x=x.view(B,Cc,H//2,2,W//2,2).permute(0,2,4,1,3,5).reshape(B,(H//2)*(W//2),Cc*4)
    return x
def unpack(x,C_,H,W):
    """(B, H/2*W/2, C*4) -> (B,C,H,W)"""
    B=x.shape[0]
    x=x.view(B,H//2,W//2,C_,2,2).permute(0,3,1,4,2,5).reshape(B,C_,H,W)
    return x
def img_ids(H,W):
    ids=torch.zeros(H//2,W//2,3,device=DEV,dtype=DT)
    ids[...,1]=torch.arange(H//2,device=DEV)[:,None]
    ids[...,2]=torch.arange(W//2,device=DEV)[None,:]
    return ids.reshape(-1,3)
print("FLUX helpers ready: pack/unpack, dynamic-shift sigmas, image ids, cached prompt")

In [ ]:
# %% F0 — 5-step SMOKE TEST. Surfaces shape/dtype errors in <1 min, not 30.
def smoke():
    from peft import LoraConfig
    pe,pp,ti=cached_prompt()
    pipe=flux_load(False)
    vae,tr,sch=pipe.vae,pipe.transformer,pipe.scheduler
    vae.requires_grad_(False); tr.requires_grad_(False)
    tr.add_adapter(LoraConfig(r=C.RANK,lora_alpha=C.RANK,init_lora_weights="gaussian",
                              target_modules=list(C.TARGETS)))
    if C.GRAD_CKPT: tr.enable_gradient_checkpointing()
    tr.train()
    ps=[p for p in tr.parameters() if p.requires_grad]
    print(f"[F0] trainable LoRA params: {sum(p.numel() for p in ps):,}")
    sf=float(vae.config.scaling_factor); sh=float(vae.config.shift_factor or 0.0)
    f=sorted(glob.glob(os.path.join(tdir("A"),"*.png")))[:4]
    px=torch.stack([torch.from_numpy(
        np.asarray(Image.open(x).convert("RGB"),np.float32)/127.5-1.0).permute(2,0,1) for x in f])
    opt=torch.optim.AdamW(ps,lr=C.LR)
    for st in range(5):
        b=px[st%len(f):st%len(f)+C.BATCH].to(DEV,dtype=DT)
        with torch.no_grad():
            lat=vae.encode(b).latent_dist.sample(); lat=((lat-sh)*sf).to(DT)
        B_,Cl,Hl,Wl=lat.shape
        seq=(Hl//2)*(Wl//2)
        if st==0:
            print(f"[F0] latent {tuple(lat.shape)}  seq_len {seq}  "
                  f"packed {(B_,seq,Cl*4)}")
            assert Cl==16, f"expected 16 latent channels, got {Cl}"
        sig_all=build_sigmas(sch,seq); n=len(sig_all)
        u=torch.sigmoid(torch.randn(B_,device=DEV,dtype=torch.float32))
        ii=(u*n).long().clamp_(0,n-1)
        sg=sig_all[ii].to(lat.dtype).view(-1,1,1,1)
        noise=torch.randn_like(lat)
        noisy=(1.0-sg)*lat+sg*noise
        pk=pack(noisy)
        ids=img_ids(Hl,Wl)
        guid=torch.full((B_,),C.GUIDANCE_TRAIN,device=DEV,dtype=DT)
        out=tr(hidden_states=pk, timestep=(sig_all[ii]).to(DT),
               guidance=guid, pooled_projections=pp.repeat(B_,1).to(DT),
               encoder_hidden_states=pe.repeat(B_,1,1).to(DT),
               txt_ids=ti.to(DT), img_ids=ids, return_dict=False)[0]
        if st==0: print(f"[F0] transformer out {tuple(out.shape)}")
        pred=unpack(out,Cl,Hl,Wl)
        loss=tF.mse_loss(pred.float(),(noise-lat).float())
        loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
        print(f"[F0] step {st+1}/5 loss {float(loss):.5f} "
              f"peak {torch.cuda.max_memory_allocated()/2**30:.1f} GB")
        if not np.isfinite(float(loss)): HALT("non-finite loss in smoke test")
    print(f"\n[F0] PASS — shapes, dtypes and gradients all sound. "
          f"Peak {torch.cuda.max_memory_allocated()/2**30:.1f} GB of "
          f"{torch.cuda.get_device_properties(0).total_memory/2**30:.0f} GB.")
    del pipe,tr,vae; gc.collect(); torch.cuda.empty_cache()
if stage("F0"): smoke()

In [ ]:
# %% F1 — train 6 adapters (3 seeds x 2 arms)
def adir(t): return os.path.join(ROOT,"adapters",t)
def _exp(t):
    _,role,seed=next(a for a in all_arms() if a[0]==t)
    return {"tag":t,"role":role,"seed":seed,"rank":C.RANK,"steps":C.STEPS,"meas":MEAS,
            "config_sha":CFG_SHA,"model":C.MODEL,"dtype":str(DT)}
def avalid(t):
    p=os.path.join(adir(t),"train_meta.json")
    if not os.path.exists(os.path.join(adir(t),"pytorch_lora_weights.safetensors")): return False
    m=json.load(open(p)) if os.path.exists(p) else None
    return m is not None and all(m.get(k)==v for k,v in _exp(t).items()) and m.get("effective",False)

def train_one(tag,pe,pp,ti):
    _,role,seed=next(a for a in all_arms() if a[0]==tag)
    if avalid(tag): print(f"[F1] skip {tag}"); return
    from peft import LoraConfig
    from peft.utils import get_peft_model_state_dict
    out=adir(tag)
    if os.path.exists(out): shutil.move(out,out+f"__stale_{int(time.time())}")
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    pipe=flux_load(False); vae,tr,sch=pipe.vae,pipe.transformer,pipe.scheduler
    vae.requires_grad_(False); tr.requires_grad_(False)
    tr.add_adapter(LoraConfig(r=C.RANK,lora_alpha=C.RANK,init_lora_weights="gaussian",
                              target_modules=list(C.TARGETS)))
    if C.GRAD_CKPT: tr.enable_gradient_checkpointing()
    tr.train()
    ps=[p for p in tr.parameters() if p.requires_grad]
    opt=torch.optim.AdamW(ps,lr=C.LR,weight_decay=1e-4)
    lrs=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=math.ceil(C.STEPS/C.GRAD_ACC))
    opt.zero_grad(set_to_none=True)
    files=sorted(glob.glob(os.path.join(tdir(role),"*.png")))
    cache=torch.stack([torch.from_numpy(
        np.asarray(Image.open(f).convert("RGB"),np.float32).copy()/127.5-1.0).permute(2,0,1)
        for f in files])
    sf=float(vae.config.scaling_factor); shf=float(vae.config.shift_factor or 0.0)
    g=torch.Generator().manual_seed(seed); losses=[]; t0=time.time(); sig_all=None; ids=None
    print(f"\n[F1] {tag}: role={role} seed={seed}, {C.STEPS} steps")
    for st in range(C.STEPS):
        idx=torch.randint(0,cache.shape[0],(C.BATCH,),generator=g)
        px=cache[idx].to(DEV,dtype=DT)
        with torch.no_grad():
            lat=vae.encode(px).latent_dist.sample(); lat=((lat-shf)*sf).to(DT)
        B_,Cl,Hl,Wl=lat.shape
        if sig_all is None:
            sig_all=build_sigmas(sch,(Hl//2)*(Wl//2)); ids=img_ids(Hl,Wl)
        n=len(sig_all)
        u=torch.sigmoid(torch.randn(B_,device=DEV,dtype=torch.float32))
        ii=(u*n).long().clamp_(0,n-1); sg=sig_all[ii].to(lat.dtype).view(-1,1,1,1)
        noise=torch.randn_like(lat)
        pk=pack((1.0-sg)*lat+sg*noise)
        guid=torch.full((B_,),C.GUIDANCE_TRAIN,device=DEV,dtype=DT)
        outp=tr(hidden_states=pk,timestep=sig_all[ii].to(DT),guidance=guid,
                pooled_projections=pp.repeat(B_,1).to(DT),
                encoder_hidden_states=pe.repeat(B_,1,1).to(DT),
                txt_ids=ti.to(DT),img_ids=ids,return_dict=False)[0]
        pred=unpack(outp,Cl,Hl,Wl)
        loss=tF.mse_loss(pred.float(),(noise-lat).float())/C.GRAD_ACC
        loss.backward()
        if (st+1)%C.GRAD_ACC==0:
            torch.nn.utils.clip_grad_norm_(ps,1.0); opt.step(); lrs.step()
            opt.zero_grad(set_to_none=True)
        losses.append(float(loss.detach())*C.GRAD_ACC)
        if not np.isfinite(losses[-1]): HALT(f"{tag}: non-finite loss at step {st+1}")
        if (st+1)%200==0:
            el=time.time()-t0
            print(f"    {st+1}/{C.STEPS} loss {np.mean(losses[-200:]):.5f} "
                  f"eta {(el/(st+1))*(C.STEPS-st-1)/60:.1f}min "
                  f"peak {torch.cuda.max_memory_allocated()/2**30:.0f}GB")
    sd=get_peft_model_state_dict(tr)
    bn=float(sum(float(v.float().norm()) for k,v in sd.items() if "lora_B" in k))
    if not (bn>1e-6 and all(bool(torch.isfinite(v).all()) for v in sd.values())):
        HALT(f"{tag}: DEAD adapter (||lora_B||={bn:.3e})")
    os.makedirs(out,exist_ok=True)
    FluxPipeline.save_lora_weights(save_directory=out,transformer_lora_layers=sd)
    m=_exp(tag); m.update({"loss_tail":float(np.mean(losses[-100:])),"lora_B_norm":bn,
                           "effective":True,"minutes":(time.time()-t0)/60})
    ajson(m,os.path.join(out,"train_meta.json"),indent=2)
    print(f"[F1] saved {tag}: ||lora_B||={bn:.2f} tail={m['loss_tail']:.5f} "
          f"({m['minutes']:.1f} min)")
    del pipe,tr,vae,cache; gc.collect(); torch.cuda.empty_cache()

if stage("F1"):
    pe,pp,ti=cached_prompt()
    for i,t in enumerate(all_tags(),1):
        print(f"[F1] adapter {i}/{len(all_tags())}: {t}"); train_one(t,pe,pp,ti)
    bad=[t for t in all_tags() if not avalid(t)]
    if bad: HALT(f"F1 incomplete: {bad}")
    print(f"[F1] PASS — {len(all_tags())} adapters")

In [ ]:
# %% F2 — generation, same paired seed bank as the base study
def gdir(t): return os.path.join(ROOT,"gens",t)
def f2():
    if DEV!="cuda": HALT("F2 needs a GPU")
    bank=[(C.CAPTION,C.GEN_SEED_BASE+j) for j in range(C.G_PER)]
    pipe=flux_load(True); pipe.set_progress_bar_config(disable=True); prev=False
    try:
        for tag in all_tags():
            d_=gdir(tag); os.makedirs(d_,exist_ok=True)
            for t_ in glob.glob(os.path.join(d_,"*.tmp")): os.remove(t_)
            miss=[i for i in range(C.G_PER) if not os.path.exists(os.path.join(d_,f"{i:05d}.png"))]
            if not miss: print(f"[F2] skip {tag}"); continue
            if prev: pipe.unload_lora_weights(); prev=False
            pipe.load_lora_weights(adir(tag)); prev=True
            print(f"[F2] {tag}: {len(miss)} missing")
            t0=time.time()
            for s in range(0,len(miss),C.GEN_BATCH):
                ix=miss[s:s+C.GEN_BATCH]
                gs=[torch.Generator(device="cpu").manual_seed(bank[i][1]) for i in ix]
                with torch.inference_mode():
                    imgs=pipe(prompt=[bank[i][0] for i in ix],height=MEAS,width=MEAS,
                              num_inference_steps=C.GEN_STEPS,guidance_scale=C.GEN_GUIDANCE,
                              generator=gs,max_sequence_length=512).images
                for i,im in zip(ix,imgs):
                    t_=os.path.join(d_,f"{i:05d}.png.tmp")
                    im.save(t_,format="PNG"); os.replace(t_,os.path.join(d_,f"{i:05d}.png"))
                if (s//C.GEN_BATCH)%25==0:
                    el=time.time()-t0
                    print(f"    {s+len(ix)}/{len(miss)}  eta "
                          f"{(el/max(s+len(ix),1))*(len(miss)-s-len(ix))/60:.0f}min")
            print(f"[F2] complete {tag} ({(time.time()-t0)/60:.0f} min)")
    finally:
        del pipe; gc.collect(); torch.cuda.empty_cache()
    print("[F2] PASS")
if stage("F2"): f2()

In [ ]:
# %% F3 — measurement against the BASE STUDY fingerprints (identical statistic)
def f3():
    K=torch.stack([torch.from_numpy(KFP[r]) for r in ("A","B")]).to(_MD)
    p=os.path.join(ROOT,"csv","f3_measure.csv"); done=done_keys(p,["tag","gen_idx","K"])
    ex=ThreadPoolExecutor(max_workers=4)
    try:
        for tag in all_tags():
            paths=[os.path.join(gdir(tag),f"{i:05d}.png") for i in range(C.G_PER)]
            todo=[i for i in range(C.G_PER) if not all((tag,str(i),r) in done for r in ("A","B"))]
            if not todo: continue
            fut={i:ex.submit(load_lum,paths[i],False) for i in todo[:6]}; t0=time.time()
            for pos,i in enumerate(todo):
                Z_np=fut.pop(i).result(); nx=pos+6
                if nx<len(todo): fut[todo[nx]]=ex.submit(load_lum,paths[todo[nx]],False)
                W=torch.from_numpy(wres(Z_np)).to(_MD); Z=torch.from_numpy(Z_np).to(_MD)
                v=nccb(W,Z.unsqueeze(0)*K).cpu().numpy()
                for j,r in enumerate(("A","B")):
                    if (tag,str(i),r) in done: continue
                    append_row(p,["tag","gen_idx","K","rho"],[tag,i,r,f"{v[j]:.6e}"])
            print(f"[F3] {tag}: {len(todo)} ({time.time()-t0:.0f}s)")
    finally:
        ex.shutdown(wait=True); gc.collect()
        if DEV=="cuda": torch.cuda.empty_cache()
    n=len(done_keys(p,["tag","gen_idx","K"])); exp=len(all_tags())*C.G_PER*2
    if n!=exp: HALT(f"F3 incomplete: {n}/{exp}")
    print(f"[F3] PASS — {n}/{exp} rows")
if stage("F3"): f3()

In [ ]:
# %% F4 — does the null replicate on a second system?
def f4():
    rng=np.random.default_rng(0)
    d=pd.read_csv(os.path.join(ROOT,"csv","f3_measure.csv"))
    def th(tag,ks,ko):
        a=d[(d.tag==tag)&(d.K==ks)].sort_values("gen_idx")["rho"].to_numpy(float)
        b=d[(d.tag==tag)&(d.K==ko)].sort_values("gen_idx")["rho"].to_numpy(float)
        n=min(len(a),len(b)); return a[:n]-b[:n]
    def sf(x,n=C.N_PERM):
        o=x.mean(); c=sum((x*rng.choice((-1.,1.),len(x))).mean()>=o for _ in range(n))
        return (c+1)/(n+1), float(o)
    out={"config_sha":CFG_SHA,"model":C.MODEL,"arms":{}}
    print(f"{'arm':5s} {'adapter':22s} {'theta':>12s} {'||lora_B||':>11s}")
    Us=[]
    for arm,ks,ko in (("A","A","B"),("B","B","A")):
        v=[]; nb=[]
        for s in C.SEEDS:
            t=f"{arm}_raw_s{s}_flux"; x=th(t,ks,ko)
            if not len(x): continue
            v.append(x)
            m=json.load(open(os.path.join(adir(t),"train_meta.json")))
            nb.append(m.get("lora_B_norm"))
            print(f"{arm:5s} {t:22s} {x.mean():+12.3e} {m.get('lora_B_norm',float('nan')):11.2f}")
        mm=np.array([x.mean() for x in v]); k=len(mm)
        p_img,_=sf(np.concatenate(v)); p_ad,_=sf(mm)
        tc=float(sps.t.ppf(1-C.ALPHA/2,df=k-1)); U=float(mm.mean()+tc*mm.std(ddof=1)/np.sqrt(k))
        Us.append(U)
        out["arms"][arm]={"n_adapters":k,"means":mm.tolist(),"theta":float(mm.mean()),
                          "sd":float(mm.std(ddof=1)),"t_crit":tc,"U_cluster":U,
                          "p_per_image":p_img,"p_adapter":p_ad,"lora_B":nb}
        print(f"      -> n={k} theta={mm.mean():+.3e} sd={mm.std(ddof=1):.3e} "
              f"t={tc:.2f} U={U:+.3e} | p_img={p_img:.4g} p_adapter={p_ad:.4g}\n")
    IUT = all(out["arms"][a]["p_adapter"]<C.ALPHA for a in ("A","B"))
    U=max(Us); out["U_device"]=U; out["IUT_pass"]=bool(IUT)
    R_real=3.56703e-02      # base-study device-specific denominator
    out["lambda_U_n3"]=U/R_real
    print("="*72)
    print(f"PRIMARY IUT (adapter level, n={out['arms']['A']['n_adapters']}): "
          f"{'PASS' if IUT else 'FAIL — the null REPLICATES on FLUX.1-dev'}")
    print(f"U_device = {U:+.4e}   lambda_U = {U/R_real:.4%}   [FLAG AS n=3, second system]")
    print(f"t_crit at n=3 is {out['arms']['A']['t_crit']:.2f} — this bound is much looser than")
    print("the primary study's n=6 result and is NOT comparable to it.")
    print("="*72)
    ajson(out,os.path.join(ROOT,"E4_flux_results.json"),indent=2)
if stage("F4"): f4()

## Running it

| session | STAGES | wall | check |
|---|---|---|---|
| 1 | `("F0",)` | ~3 min | shapes printed, 5 finite losses, peak VRAM well under 96 GB |
| 2 | `("F1",)` | ~4–6 h | every ‖lora_B‖ non-zero |
| 3 | `("F2",)` | ~6–8 h | 3000 images at 1024, 28 steps |
| 4 | `("F3","F4")` | ~20 min | **does the null replicate** |

**Do not skip F0.** It exercises the packing, the dynamic-shift sigma schedule, the guidance
embedding and the three text inputs in five steps. If any of them is wrong you find out in under a
minute.

**Reading F4.** A FAIL on the IUT is the expected and desired outcome — it means the null replicates
on a second generative system whose autoencoder transmits 1.5× more device signal than the
incumbent. Report the bound flagged as **n = 3, second system**: t₀.₉₉₅,₂ = 9.92 makes it far
looser than the primary n = 6 result, and the two are not comparable.

**If F0 fails**, the likely culprits in order: `encode_prompt` returning a different tuple arity in
your diffusers version; `txt_ids`/`img_ids` expecting 2-D rather than 3-D; or the transformer
wanting `timestep` scaled differently. Send me the traceback rather than guessing — each is a
one-line fix.